In [1]:
from twelve import StockTicker
import pandas as pd

Time Setting


In [50]:
timeframe = '1day'
date_start = '2004-08-19'
date_end = '2013-03-01'

#For the example take:
#timeframe = '1day'
#date_start = '2004-08-19'
#date_end = '2013-03-01'

#### Data Extraction and Cleaning

In [43]:
##RUN ONLY ONCE! Can Block API.
#Companies you want to get data for; Time frame of the data;
tickers1 = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ','GOOG']#, 'AMZN', 'CAT', 'PG', 'NVDA', 'KO']

stocks1 = []
data1 = {}

for j in range(len(tickers1)):
    #Calling the Class StockTicker to get the data for each ticker in the list tickers1.
    stock = StockTicker(symbol=tickers1[j], interval=timeframe, start_date=date_start, end_date=date_end)
    stocks1.append(stock)

    #Taking the data from each each ticker, then formatting it such that the (pandas) index is the time as required by the backtesting library https://kernc.github.io/backtesting.py/doc/examples/Quick%20Start%20User%20Guide.html
    data1[tickers1[j]] = stock.get_data(show=False)
    data1[tickers1[j]].columns = data1[tickers1[j]].columns.str.capitalize()
    data1[tickers1[j]][["Open", "High", "Low", "Close", "Volume"]] = data1[tickers1[j]][["Open", "High", "Low", "Close", "Volume"]].apply(pd.to_numeric)
    data1[tickers1[j]]["Datetime"] = pd.to_datetime(data1[tickers1[j]]["Datetime"])
    data1[tickers1[j]].set_index("Datetime", inplace=True)


In [44]:
#Saves Data as CSV

for j in range(len(tickers1)):
    data1[tickers1[j]].to_csv(f"twelve_data_csv/{tickers1[j]}.csv")

#can then later be read as:
#AAPL = pd.read_csv(f"{twelve_data_csv}/AAPL.csv", index_col="Datetime", parse_dates=True)


### Example: Moving Average Strategy as presented in: https://kernc.github.io/backtesting.py/doc/examples/Quick%20Start%20User%20Guide.html
We code in this strategy to see the workings of the backtesting system, while also using it as a potential metric against other strategies. Note: This code is mostly copy and pasted, except we apply it to our data.
This also guides us to how we should generally formulate/present our strategies with a simple example.

In [ ]:
from backtesting import Strategy
from backtesting.lib import crossover
from backtesting import Backtest

#Simple Moving Average (SMA) function
def SMA(values, n):
    """
    Return simple moving average of `values`, at
    each step taking into account `n` previous values.
    """
    return pd.Series(values).rolling(n).mean()

#STRATEGY: SMA Crossover. General idea is to buy when the short-term SMA crosses above the long-term SMA, and sell when the short-term SMA crosses below the long-term SMA.
class SmaCross(Strategy):
    # Define the two MA lags as *class variables*
    # for later optimization
    n1 = 10
    n2 = 20
    
    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # If sma1 crosses above sma2, close any existing
        # short trades, and buy the asset
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()

        # Else, if sma1 crosses below sma2, close any existing
        # long trades, and sell the asset
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()
            
bt = Backtest(data1['GOOG'], SmaCross, cash=10_000, commission=.002)
stats = bt.run()
stats

C:\Users\nikit\AppData\Local\Temp\ipykernel_15544\2885104948.py:37: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt = Backtest(data1['MSFT'], SmaCross, cash=10_000, commission=.002)
C:\Users\nikit\AppData\Local\Temp\ipykernel_15544\2885104948.py:38: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Start                     2004-08-19 00:00:00
End                       2013-02-28 00:00:00
Duration                   3115 days 00:00:00
Exposure Time [%]                    97.90405
Equity Final [$]                   9131.35141
Equity Peak [$]                   17642.36716
Commissions [$]                    5379.58397
Return [%]                           -8.68649
Buy & Hold Return [%]                 1.98092
Return (Ann.) [%]                    -1.06141
Volatility (Ann.) [%]                26.33093
CAGR [%]                             -1.05986
Sharpe Ratio                         -0.04031
Sortino Ratio                        -0.05804
Calmar Ratio                         -0.02157
Alpha [%]                            -8.41697
Beta                                 -0.13605
Max. Drawdown [%]                   -49.20868
Avg. Drawdown [%]                    -5.91269
Max. Drawdown Duration      995 days 00:00:00
Avg. Drawdown Duration       93 days 00:00:00
# Trades                          

In [46]:
bt.plot()

GridPlot(id='p3761', ...)

In [ ]:
#Now we do an optimization of our dependnet variables n1 and n2, with the goal of increasing the amount of equity at the end.

stats = bt.optimize(n1=range(5, 30, 5),
                    n2=range(10, 70, 5),
                    maximize='Equity Final [$]',
                    constraint=lambda param: param.n1 < param.n2)

stats

c:\Users\nikit\Projects\tr_al\venv\Lib\site-packages\backtesting\backtesting.py:1639: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()
c:\Users\nikit\Projects\tr_al\venv\Lib\site-packages\backtesting\backtesting.py:1652: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
c:\Users\nikit\Projects\tr_al\venv\Lib\site-packages\backtesting\backtesting.py:1652: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finaliz

Start                     2004-08-19 00:00:00
End                       2013-02-28 00:00:00
Duration                   3115 days 00:00:00
Exposure Time [%]                    95.76153
Equity Final [$]                   19977.6333
Equity Peak [$]                   26398.76289
Commissions [$]                    3775.08516
Return [%]                           99.77633
Buy & Hold Return [%]                -1.31345
Return (Ann.) [%]                     8.46565
Volatility (Ann.) [%]                27.32643
CAGR [%]                               8.4527
Sharpe Ratio                           0.3098
Sortino Ratio                          0.5102
Calmar Ratio                          0.25897
Alpha [%]                            99.54155
Beta                                 -0.17876
Max. Drawdown [%]                    -32.6893
Avg. Drawdown [%]                       -4.33
Max. Drawdown Duration      974 days 00:00:00
Avg. Drawdown Duration       48 days 00:00:00
# Trades                          

In [49]:
bt.plot(plot_volume=False, plot_pl=False)

GridPlot(id='p4249', ...)

In [2]:
import requests
from decouple import config

api_key = config("POLYGON_KEY")

url = "https://api.polygon.io/v3/reference/options/contracts"

wanted_strikes = [7000, 7200, 7400, 7600, 7800, 8000]

selected_contracts = []

for strike in wanted_strikes:

    params = {
        "underlying_ticker": "SPX",
        "contract_type": "call",
        "expiration_date": "2026-12-18",
        "strike_price": strike,
        "limit": 10,
        "apiKey": api_key,
    }

    response = requests.get(url, params=params)

    print("Strike:", strike, "Status:", response.status_code)

    if response.status_code == 200:
        data = response.json()

        for c in data.get("results", []):
            selected_contracts.append(c)

Strike: 7000 Status: 200
Strike: 7200 Status: 200
Strike: 7400 Status: 200
Strike: 7600 Status: 200
Strike: 7800 Status: 200
Strike: 8000 Status: 429
